# Barnase–barstar walkthrough

A guided tour of the Phase-1 pipeline on the classic **barnase–barstar** complex (PDB `1BRS`) — the most-measured protein interface in existence, and our known-answer test case.

**What you'll see:** we detect the interface, compute per-residue features, and rank hot-spot candidates. The chemistry-aware ranking recovers the textbook hot spots — and pulls barnase **Arg87** from buriedness rank #38 up to the top few, exactly the way *interaction chemistry* rescues a residue that *burial* would miss. That's the whole thesis of the project, reproduced on a complex where the answer is known.

> Biology background lives in [`docs/biology/`](../docs/biology/00_overview.md). Read `00`–`02` alongside this.

In [ ]:
# If running from a fresh clone, make the src package importable and (in Colab) install deps.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
# In Colab, uncomment:
# !pip install biopython numpy scipy pandas py3Dmol requests truststore

from hotspot.pipeline import analyze_complex
from hotspot.report import text_report, contacts_to_frame
import pandas as pd
pd.set_option('display.max_columns', None); pd.set_option('display.width', 200)

## 1. Run the whole pipeline in one call

`1BRS` contains three copies of the complex (chains A/B/C = barnase, D/E/F = barstar). We analyze one copy: **barnase A vs barstar D**. The structure downloads automatically the first time (cached in `data/raw/`).

In [ ]:
analysis = analyze_complex('1BRS', chains='A,D')
print(f'{len(analysis.interface)} interface residues, {len(analysis.contacts)} interactions detected')
print('SASA backend:', analysis.sasa_backend)

## 2. The ranked hot-spot table

One row per interface residue, ranked by the chemistry-aware `hotspot_score`. Compare `hotspot_rank` to `naive_rank` (buriedness only).

In [ ]:
analysis.top(10)

## 3. The money shot: where chemistry and buriedness disagree

Barnase **Arg87** is a genuine, experimentally-confirmed hot spot, but it isn't heavily buried — so naive buriedness ranks it far down the list. Its salt-bridge chemistry pulls it up. This is the barnase–barstar analogue of the Y96-vs-Arg95 story.

In [ ]:
t = analysis.table
movers = t[['residue','aa','hotspot_rank','naive_rank','hotspot_score','dsasa','n_salt_bridges','reasoning']].copy()
movers['rank_jump'] = movers['naive_rank'] - movers['hotspot_rank']  # positive = chemistry promoted it
movers.sort_values('rank_jump', ascending=False).head(8)

## 4. Every detected interaction (validate against LigPlot+/DIMPLOT)

The atom-level contact list. Run DIMPLOT on 1BRS and check it finds the same H-bonds and hydrophobic contacts — an independent check on our geometry code.

In [ ]:
cdf = contacts_to_frame(analysis)
print(cdf['kind'].value_counts().to_string())
cdf[cdf['kind'] == 'salt_bridge']

## 5. See it in 3D

The complex as a cartoon; interface residues as thin sticks; the top hot spots as fat, colored sticks with labels. Rotate it.

In [ ]:
from hotspot.viz import show_interface
show_interface(analysis, top_n=5)

## 6. Full text report

The same summary the CLI prints and saves to `outputs/`.

In [ ]:
print(text_report(analysis, top_n=10))

---
### Where this goes next
- Wire in **conservation** (highest-value remaining feature).
- Cross-check contacts vs. **DIMPLOT**, and cutoffs with **Sandra**.
- Point the tool at **ARL15–CNNM2**: does it rank **Arg95** above **Y96**?
- **Phase 2:** run this pipeline across **SKEMPI 2.0**, join ΔΔG labels, train XGBoost → GNN, and ablate to show which features carry the signal. See [`docs/roadmap.md`](../docs/roadmap.md).